# Sklearn Pipelines Exercise
*Made by viga@itu.dk and thso@itu.dk*

## Introduction

In this exercise you'll be working with the [Wine Quality Dataset](https://archive.ics.uci.edu/ml/datasets/wine+quality) from the UCI Machine Learning Repository. The dataset consists of 11 features and a quality score for 4898 white wine samples and 1599 red wine samples. The goal is to predict the quality of the wine based on the features.

The datasets are located in the `data` folder. The `winequality-red.csv` file contains the red wine samples and the `winequality-white.csv` file contains the white wine samples. Lastly, the `winequality.names` file contains a description of the dataset.

The goal of this exercise is to get you familiar with the [Scikit-learn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) API. You'll be using pipelines to perform feature scaling and feature selection.

## Load in the data

You can either load the red-wine dataset or the white-wine dataset. You can also load both datasets and combine them if you want.

Both datasets are available in the `data` folder, and are called `winequality-red.csv` and `winequality-white.csv`.

Hint: You can use the `pd.read_csv()` function to load in the data (remember to check the delimiter!). You can find the documentation [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html).


In [5]:
import pandas as pd
# load in the data
red_wine = pd.read_csv("data/winequality-red.csv", sep=";")
white_wine = pd.read_csv("data/winequality-white.csv", sep=";")

# if you want to you can combine the two datasets into one - but this is not necessary
red_wine["wine_type"] = "red"
white_wine["wine_type"] = "white"
wine_data = pd.concat([red_wine, white_wine], ignore_index=True)

# check a few rows of the data - hint: use .head()
wine_data.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


## Data Exploration

### Check the number of missing values in the dataset.

Hint: `.isnull()`

Dont worry if there are missing values, we'll handle them later in our pipeline!

In [6]:
wine_data.isnull().sum()

fixed acidity           3
volatile acidity        5
citric acid             8
residual sugar          6
chlorides               7
free sulfur dioxide     6
total sulfur dioxide    6
density                 3
pH                      5
sulphates               6
alcohol                 6
quality                 0
wine_type               0
dtype: int64

### Check some basic statistics

We want to know the mean, standard deviation, minimum, maximum and quartiles of each feature.
This will give us a good idea of the distribution of the data, and also tell us if we need to do any scaling.

Hint: `.describe()`, If the output is hard to read, you can use `.T` to transpose the dataframe, i.e., swapping the rows and columns.

Do you notice anything strange about the data? Is there anything that stands out to you?

In [ ]:
# check some basic statistics
wine_data.describe().T

# What we notice: they all have different ranges. We will need to scale (standardize). 

,count,mean,std,min,25%,50%,75%,max
fixed acidity,6494.0,7.215745,1.296537,3.80000,6.40000,7.00000,7.70000,15.90000
volatile acidity,6492.0,0.339586,0.164520,0.08000,0.23000,0.29000,0.40000,1.58000
citric acid,6489.0,0.318550,0.145211,0.00000,0.25000,0.31000,0.39000,1.66000
residual sugar,6491.0,5.442782,4.758549,0.60000,1.80000,3.00000,8.10000,65.80000
chlorides,6490.0,0.056045,0.035046,0.00900,0.03800,0.04700,0.06500,0.61100
free sulfur dioxide,6491.0,30.529965,17.747755,1.00000,17.00000,29.00000,41.00000,289.00000
total sulfur dioxide,6491.0,115.762209,56.532631,6.00000,77.25000,118.00000,156.00000,440.00000
density,6494.0,0.994696,0.002999,0.98711,0.99234,0.99489,0.99699,1.03898
pH,6492.0,3.218490,0.160781,2.72000,3.11000,3.21000,3.32000,4.01000
sulphates,6491.0,0.531228,0.148832,0.22000,0.43000,0.51000,0.60000,2.00000


We saw that there were some missing values in the dataset, this we can fix in the pipeline, using the [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) from sklearn.

Next we also saw that there was a some differences in the scale of the different variables, so we will use the [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) from sklearn to scale the data. This will make it easier for the model to learn the patterns in the data. Especially for the KNN algorithm (which we'll use), which is based on distance, it is important that the data is scaled.

If you think of other transformations that might be useful for this dataset, feel free to try them out!

**Take a look at the [sklearn.preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html) module for some inspiration.**

# Splitting the data

Now that we have created our pipeline, we can train the model.

First we need to split the data into a training set and a test set. We will use the [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function from sklearn to do this. But first we need to split the data into features and labels.

The features are all the columns in the dataset, except for the `quality` column, which are the labels.

We will use the default split of 75% training data, and 25% test data.

Hint: You can use the `random_state` parameter to make sure that the data is split the same way every time you run the code.

The train_test_split function returns four values, the first two are the training and test data, and the last two are the train and test labels.

In [ ]:
from sklearn.model_selection import train_test_split
# split the data into X and y
X = wine_data.drop(columns=["quality"]) # We drop the quality column, as we will use this as the y (the answers - what we want to predict)
y = wine_data["quality"] # now y is the "answers"

# now split the data into train and test data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42) # the random state makes sure the split is reproducable.
# Default split: 75% train and 25% test

#   We can shuffle sinze there is no temporal order... Default makes it shuffle so we do not have to specify it.
#   since red are first and white after, it will actually be a good thing to shuffle, such that we do not only test on one type of wine. 

X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,wine_type
1539,7.2,0.390,0.32,1.80,0.065,34.0,60.0,0.99714,3.46,0.78,9.9,red
1109,10.8,0.470,0.43,2.10,0.171,27.0,66.0,0.99820,3.17,0.76,10.8,red
100,8.3,0.610,0.30,2.10,0.084,11.0,50.0,0.99720,3.40,0.61,10.2,red
5477,6.5,0.350,0.31,10.20,0.069,58.0,170.0,0.99692,3.18,0.49,9.4,white
6416,5.8,0.385,0.25,3.70,0.031,38.0,122.0,0.99128,3.20,0.63,11.2,white
...,...,...,...,...,...,...,...,...,...,...,...,...
3772,7.6,0.320,0.58,16.75,0.050,43.0,163.0,0.99990,3.15,0.54,9.2,white
5191,5.6,0.280,0.27,3.90,0.043,52.0,158.0,0.99202,3.35,0.44,10.7,white
5226,6.4,0.370,0.20,5.60,0.117,61.0,183.0,0.99459,3.24,0.43,9.5,white
5390,6.5,0.260,0.50,8.00,0.051,46.0,197.0,0.99536,3.18,0.47,9.5,white


# Creating the pipeline

We will now create a pipeline that will handle the missing values and scaling for us, and finally train a KNN model on the data.

We will use the [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) class from sklearn to create our pipeline.

The pipeline will consist of three steps, the first step will be to impute the missing values, and the second step will be to scale the data, and the third step will be to train the model.

The pipeline format is a list of tuples, where the first element in the tuple is the name of the step, and the second element is the step itself, e.g.:

```python
pipeline = Pipeline([
	('step_name', step()),
	('step_name', step()),
	('step_name', step()),
])
```

Where the `step_name` is a string, and the `step` is a sklearn object - this can be a "Transformer" object (like `SimpleImputer` and `StandardScaler`) or an "Estimator" object (like `KNeighborsClassifier` or `LinearRegression`).

In [12]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

# create your pipeline
pipe = Pipeline([
    
])

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_features = X.select_dtypes(include="number").columns
categorical_features = ["wine_type"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scalar", StandardScaler())
    ]), numeric_features),
    ("cat", OneHotEncoder(), categorical_features)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("knn", KNeighborsClassifier())
])

# Evaluating the model

Now that we have trained the model, we want to evaluate it to see how well it performs.

We will use the [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) function from sklearn to calculate the accuracy of the model.

Since we have created a pipeline, we can simply call the `.fit()` and `.predict()` methods on the pipeline object, and it will handle the preprocessing for us - and importantly in the correct order.

Remember to only call `.fit()` on the training data. Calling `.fit()` on the test data will cause the model to overfit to the test data, and will give you an overly optimistic accuracy score.

* **`.fit(X_train, y_train)` will train the model on the training data.**
* **`.predict(X_train)` will return the predicted labels for the test data, which you can then pass to the `accuracy_score` function, along with the true labels (y_train).**
* **`.predict(X_test)` will return the predicted labels for the test data, which you can then pass to the `accuracy_score` function, along with the true labels (y_test).**

In [ ]:
from sklearn.metrics import accuracy_score
#Start logging before training/fitting the model
#mlflow.autolog()

# fit the pipeline
pipeline.fit(X_train, y_train)

# evaluate the pipeline
y_pred = pipeline.predict(X_test)
print("Train accuracy: ", accuracy_score(y_train, pipeline.predict(X_train)))
print("Test accuracy: ", accuracy_score(y_test, y_pred))

Train accuracy:  0.7040229885057471
Test accuracy:  0.5415384615384615
